# Statistical mechanics of money

The **Boltzmann-Gibbs (BG) distribution** is a fundamental concept in statistical mechanics, describing the equilibrium distribution of energy among particles in a closed system. This distribution can also be applied to economic models, where it characterizes the equilibrium distribution of money among agents. 

Mathematically, the BG distribution is expressed as:
$$
P(m) = \frac{1}{T} \exp\left(-\frac{m}{T}\right),
$$
where $P(m)$ represents the probability density for an agent to hold money $m$, and $T$ is the effective "temperature" of the system. In economic terms, $T$ is proportional to the average money per agent, defined as:
$$
T = \frac{M}{N},
$$
where $M$ is the total money in the system, and $N$ is the number of agents.

The BG distribution exhibits an exponential decay, implying that higher values of money are less probable. This behavior reflects the tendency of systems in statistical equilibrium to maximize entropy while conserving total resources. The emergence of the BG distribution relies on key assumptions, such as the conservation of money, random transactions between agents, and symmetry in exchange rules.

In the context of economic models, the BG distribution serves as a baseline for understanding the equilibrium states of systems with randomized money exchange. It allows for the analysis of deviations caused by additional dynamics, such as taxation, saving behavior, or asymmetries in transaction rules. In this project, we investigate how the BG distribution emerges in different scenarios and explore the conditions under which deviations occur, providing insights into the underlying mechanisms of statistical equilibrium.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import functions

# Simulation Without Debt

This simulation models a closed economic system where agents exchange money without borrowing (no debt allowed). The total money $M$ in the system remains conserved across all transactions.

### Agent Initialization
- **Number of agents $N$:** Each agent starts with an equal share of the total money $\frac{M}{N}$.
- **Money conservation:** No new money is introduced, and no debt is allowed.

### Transaction Mechanisms
The type of transaction is controlled by the `transaction_type` parameter:
1. **Constant:** A fixed amount of 1 unit is exchanged between two agents.
2. **Fraction of Pair's Average:** A random fraction of the average money of the pair $\frac{m_i + m_j}{2}$ is exchanged.
   $$
   \Delta = r \cdot \frac{m_i + m_j}{2}, \quad r \sim U(0, 1)
   $$
3. **Fraction of System Average:** A random fraction of the system-wide average money $\langle m \rangle$ is exchanged.
   $$
   \Delta = r \cdot \langle m \rangle, \quad r \sim U(0, 1)
   $$

### Conservation of Money
- For each transaction:
  $$
  m_i' = m_i - \Delta, \quad m_j' = m_j + \Delta
  $$
  where $m_i$ and $m_j$ are the money held by agents $i$ and $j$, respectively, before the transaction.

- Transactions only occur if $m_i \geq \Delta$. Otherwise, they are skipped.


# Simulation Results: Final Distribution Without Debt

### Boltzmann-Gibbs Distribution
The stationary distribution of money, according to the Boltzmann-Gibbs law, is given by:
$$
P(m) = \frac{1}{T} \exp\left(-\frac{m}{T}\right)
$$
where:
- $P(m)$ is the probability density for money $m$.
- $T$ is the system's "temperature," defined as the mean money per agent:
  $$
  T = \langle m \rangle = \frac{M}{N}.
  $$

### Steps
1. **Simulation**:
   - Use the `simulate_no_debt` function to model the system for $N=500$ agents, $M=5 \times 10^5$ total money, and `transaction_type='fraction_system'`.
   - The results are stored in `final_agents_no_debt`.

3. **Boltzmann-Gibbs Fit**:
   - Fit the histogram with the theoretical Boltzmann-Gibbs law using $T = \langle m \rangle$.

4. **Log-Log Plot**:
   - Transform the data to a logarithmic scale:
     $$
     \log(P(m)) = -\frac{m}{T} + \log\left(\frac{1}{T}\right).
     $$
   - Plot the transformed data and fit it against the logarithmic form of the Boltzmann-Gibbs law.

In [4]:
# Simulation without debt
N = 500
M = 5e5
final_agents_no_debt = functions.simulate_no_debt(N=N, M=M, steps=5e5, transaction_type='fraction_system')
money_dist_no_debt, bins_no_debt = np.histogram(final_agents_no_debt, bins=50, density=True)

# BG Fit: P(m) = (1/T) * exp(-m/T), T = mean
T_no_debt = np.mean(final_agents_no_debt)
m_centers_no_debt = 0.5 * (bins_no_debt[1:] + bins_no_debt[:-1])
fit_no_debt = (1 / T_no_debt) * np.exp(-m_centers_no_debt / T_no_debt)

In [6]:
final_agents_no_debt = functions.simulate_no_debt(N=500, M=5e5, steps=5e5, transaction_type='fraction_system')
counts, bin_edges = np.histogram(final_agents_no_debt, bins=50, density=True)
bin_width = bin_edges[1] - bin_edges[0]
total_probability = np.sum(counts * bin_width)

print("Probabilities sum (~1):", total_probability)

Probabilities sum (~1): 1.0


# Entropy Evolution Over Time

The **entropy** of the money distribution evolves over time in a system without debt. Entropy measures the level of disorder or randomness in the system and is a key indicator of equilibrium.

### Entropy Definition
The entropy of the system is defined as: $ S = -\sum_{i} p_i \ln(p_i), $
where:
- $p_i$ is the probability density for money in bin i,
- $S$ increases as the system becomes more disordered.

The rate at which entropy increases depends on the transaction rule:
- For the **constant exchange rule**, mixing is much slower, and it takes significantly longer for the system to approach equilibrium.
- For the **fractional system exchange rule**, mixing is more efficient, allowing the system to reach equilibrium and maximum entropy more quickly.

This behavior reflects the link between the efficiency of redistribution and the time required for the system to achieve statistical equilibrium.

### Key Observations
- The saturation of entropy implies the system has reached **statistical equilibrium**.
- The initial rapid increase reflects the redistribution of money among agents starting from a highly ordered state (equal distribution).

# Simulation with Debt

This simulation extends the basic money exchange model by allowing agents to incur debt up to a maximum limit, $-md$. This modification introduces a new dynamic in the system, where agents can spend more money than they have, enabling different outcomes in the distribution of wealth.

### Key Parameters
1. **Maximum debt ($md$):**
   - Each agent can go into debt up to $-md$.
2. **Transaction types:**
   - **Constant:** Fixed amount of 1 unit exchanged.
   - **Fraction of Pair's Average:** A random fraction of the average money of the pair, $ \Delta = r \cdot \frac{m_i + m_j}{2}, \, r \sim U(0, 1)$.
   - **Fraction of System Average:** A random fraction of the system-wide average money, $ \Delta = r \cdot \langle m \rangle, \, r \sim U(0, 1)$.

### Transaction Rules
1. A transaction occurs between two random agents $i$ and $j$.
2. If agent $i$ has enough money, the transaction proceeds as:
   $$
   m_i' = m_i - \Delta, \quad m_j' = m_j + \Delta.
   $$
3. If agent $i$ does not have enough money, they can still complete the transaction provided their new balance does not fall below $-md$:
   $$
   m_i' \geq -md.
   $$

### Conservation of Total Money
Despite the introduction of debt, the **total money in the system remains conserved**:
$$
\sum_{i=1}^N m_i = M.
$$

### Observations
- Introducing debt can result in a broader distribution of money, as agents can temporarily "overspend."
- The maximum debt $md$ acts as a control parameter for the system's behavior and wealth inequality.


# Final Distribution with Debt

This simulation examines the final stationary distribution of money among agents **with debt**. The inclusion of debt introduces a new dynamic to the system, allowing agents to temporarily have negative balances up to a maximum debt limit, $-md$.

### Key Parameters
- **Maximum debt ($md$):** The maximum negative balance an agent can hold.
- **Temperature correction ($T$):**
  - When debt is allowed, the effective "temperature" of the system increases:
    $$
    T = \frac{M}{N} + md,
    $$
    where:
    - $M$ is the total money,
    - $N$ is the number of agents,
    - $md$ is the maximum debt limit.

### Boltzmann-Gibbs Distribution
The stationary distribution with debt still follows the Boltzmann-Gibbs form:
$$
P(m) = \frac{1}{T} \exp\left(-\frac{m}{T}\right),
$$
where $T$ is the corrected temperature.

### Steps
1. **Simulation**:
   - Agents exchange money using the `simulate_with_debt` function, with $N=500$, $M=5 \times 10^5$, $steps=4 \times 10^5$, and $md=800$.
   - Transactions allow agents to go into debt, provided their balance does not fall below $-md$.

2. **Fit**:
   - Fit the histogram using the Boltzmann-Gibbs law, with the corrected temperature $T = \frac{M}{N} + md$.

3. **Visualization**:
   - The histogram represents the simulated distribution of money.
   - The red curve shows the Boltzmann-Gibbs fit for the given temperature $T$.

### Observations
- The Boltzmann-Gibbs fit remains valid, demonstrating that the system reaches a new equilibrium determined by the effective temperature.

In [8]:
# Simulation with debt
md = 800
final_agents_debt = functions.simulate_with_debt(N=N, M=M, steps=4e5, md=md, transaction_type='fraction_system')
money_dist_debt, bins_debt = np.histogram(final_agents_debt, bins=50, density=True)

m_centers_debt = 0.5 * (bins_debt[1:] + bins_debt[:-1])

# Calculate the corrected temperature with debt
T_debt = M / N + md
# Boltzmann-Gibbs Fit
fit_debt = (1 / T_debt) * np.exp(-m_centers_debt / T_debt)

# COMPARISON: DEBT VS NO DEBT MODELS